# Droplet-coalescence analysis

Reads the silo output from this case directory and visualizes the mesh, the interface, and the actual per-cell $\alpha_1$ values before reporting any derived scalars.

Order of cells:

1. Config, helpers
2. Discover timesteps, load the target step
3. Cells per diameter on each axis
4. Midplane $\alpha_1$ field, cell-by-cell (`pcolormesh` with mesh lines)
5. Close-up at the outer face of the right droplet, with per-cell $\alpha$ values annotated where $0.05 < \alpha < 0.95$
6. $\alpha_1$ sampled along an x-ray through each droplet centre, plotted as discrete per-cell values with cell boundaries marked
7. Interface thickness (from the same ray) and a $|\nabla \alpha|$ cross-check, printed after the plots
8. Time series (skipped when only one step exists)
9. Summary

**Prerequisite:** `silo_hdf5/` must exist in this directory (run `./mfc.sh run case.py --case a` with `post_process` enabled, format=1).

**Python env:** must have `h5py`, `numpy`, `matplotlib`. The MFC toolchain venv works:

```
source ../../build/venv/bin/activate
jupyter lab analysis.ipynb
```

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MFC_ROOT = NOTEBOOK_DIR.parents[1]
sys.path.insert(0, str(MFC_ROOT / "toolchain"))

import numpy as np
import matplotlib.pyplot as plt
from mfc.viz import assemble_silo, discover_timesteps

D = 240e-6
R = D / 2.0

# Mirror of the CASES table in case.py. Keep in sync if case.py changes.
CASES = {
    "a": (0.2, 14.8, 0.20),  "b": (0.5, 23.6, 0.10),  "c": (8.6, 105.9, 0.08),
    "d": (15.2, 139.8, 0.08), "e": (19.4, 158.0, 0.05), "f": (32.8, 210.8, 0.08),
    "g": (37.2, 228.0, 0.01), "h": (61.4, 296.5, 0.06), "i": (61.3, 295.3, 0.11),
    "j": (56.3, 288.9, 0.13), "k": (70.8, 327.7, 0.25), "l": (48.1, 270.1, 0.39),
    "m": (60.1, 302.8, 0.55), "n": (65.1, 320.3, 0.49), "o": (60.8, 313.7, 0.68),
    "p": (64.9, 312.8, 0.71), "q": (48.8, 260.3, 0.72), "r": (14.5, 149.1, 0.34),
}

CASE_LABEL = "a"
_, _, B = CASES[CASE_LABEL]
sep = 0.55 * D
b_impact = B * D

DROPLETS = {
    "left":  (-sep, -b_impact / 2.0, 0.0),
    "right": (+sep, +b_impact / 2.0, 0.0),
}

CASE_DIR = str(NOTEBOOK_DIR)
STEP = None


def interface_thickness_x_ray(alpha, x_cc, y_cc, z_cc, xc, yc, zc, R):
    jy = int(np.argmin(np.abs(y_cc - yc)))
    kz = int(np.argmin(np.abs(z_cc - zc)))
    if xc >= 0:
        mask = (x_cc >= xc) & (x_cc <= xc + 2.0 * R)
    else:
        mask = (x_cc <= xc) & (x_cc >= xc - 2.0 * R)

    x_seg = x_cc[mask]
    a_seg = alpha[mask, jy, kz]

    if a_seg.min() > 0.05 or a_seg.max() < 0.95:
        raise RuntimeError(
            f"alpha range on ray = [{a_seg.min():.3g}, {a_seg.max():.3g}]; "
            "interface not fully captured on this ray"
        )

    order = np.argsort(a_seg)
    x_05 = float(np.interp(0.05, a_seg[order], x_seg[order]))
    x_95 = float(np.interp(0.95, a_seg[order], x_seg[order]))
    t_m = abs(x_95 - x_05)
    dx_local = float(np.mean(np.diff(x_seg)))
    return t_m, t_m / dx_local, x_seg, a_seg, x_05, x_95

In [ ]:
steps = discover_timesteps(CASE_DIR, "silo")
if not steps:
    raise RuntimeError(
        f"No silo timesteps found under {CASE_DIR!r}. "
        "Run pre_process + post_process (format=1) first."
    )

step = STEP if STEP is not None else steps[0]
if step not in steps:
    raise ValueError(f"step={step} not in available steps {steps}")

print(f"Available steps ({len(steps)}): {steps[:10]}{' ...' if len(steps) > 10 else ''}")
print(f"Using step: {step}")

data = assemble_silo(CASE_DIR, step)
print(f"ndim = {data.ndim}")
print(f"grid = {len(data.x_cc)} x {len(data.y_cc)} x {len(data.z_cc)}")
print(f"variables = {sorted(data.variables)}")

ALPHA_NAME = next((n for n in ("alpha1", "alpha_1") if n in data.variables), None)
if ALPHA_NAME is None:
    raise KeyError(f"No alpha1-like variable in {sorted(data.variables)}")
alpha = data.variables[ALPHA_NAME]
x, y, z = data.x_cc, data.y_cc, data.z_cc
print(f"alpha: {ALPHA_NAME}  shape={alpha.shape}  range=[{alpha.min():.3g}, {alpha.max():.3g}]")

In [ ]:
def _axis_stats(coord):
    d = np.diff(coord)
    return d.mean(), d.min(), d.max()

stats = {
    "x": (len(x), *_axis_stats(x)),
    "y": (len(y), *_axis_stats(y)),
    "z": (len(z), *_axis_stats(z)),
}

print(f"{'axis':<5}{'N':>6}{'Δ [um]':>12}{'CPD = D/Δ':>14}{'nonuniformity':>18}")
print("-" * 55)
for name, (n, d, dmin, dmax) in stats.items():
    u = (dmax - dmin) / max(dmin, 1e-300)
    tag = "uniform" if u < 1e-6 else f"{u:.1e}"
    print(f"{name:<5}{n:>6d}{d*1e6:>12.4f}{D/d:>14.2f}{tag:>18}")

dx = stats["x"][1]
dy = stats["y"][1]
dz = stats["z"][1]

## Midplane $\alpha_1$ field on $z=0$

`pcolormesh` with `shading="nearest"` renders each grid cell as a discrete coloured quad centred on its cell-centre coordinate. At full 400x280 resolution the mesh lines look like noise; zoom in (toolbar or `ax.set_xlim / set_ylim`) to see them resolve into cells.

In [ ]:
kz0 = int(np.argmin(np.abs(z - 0.0)))
slice_xy = alpha[:, :, kz0]

fig, ax = plt.subplots(figsize=(10, 6))
pc = ax.pcolormesh(
    x / D, y / D, slice_xy.T,
    shading="nearest",
    cmap="viridis",
    vmin=0.0, vmax=1.0,
    edgecolors="k",
    linewidths=0.05,
)
ax.contour(x / D, y / D, slice_xy.T, levels=[0.5], colors="r", linewidths=0.8)
for label, (xc, yc, _zc) in DROPLETS.items():
    ax.plot(xc / D, yc / D, "r+", ms=12, mew=2)
ax.set_xlabel("x / D")
ax.set_ylabel("y / D")
ax.set_aspect("equal")
ax.set_title(f"$\\alpha_1$ on z=0 midplane (step {step})")
fig.colorbar(pc, ax=ax, label=r"$\alpha_1$")
plt.show()

## Interface close-up

Zoom onto the outer face of the **right** droplet on the $z=0$ midplane. Cell edges are drawn, and the per-cell $\alpha_1$ value is printed inside every cell where $0.05 < \alpha_1 < 0.95$ (the interface band). Red line is the $\alpha_1 = 0.5$ isocontour.

In [ ]:
xc_r, yc_r, _zc_r = DROPLETS["right"]
x_lo, x_hi = xc_r + 0.2 * R, xc_r + 1.2 * R
y_lo, y_hi = yc_r - 0.5 * R, yc_r + 0.5 * R

mx = (x >= x_lo) & (x <= x_hi)
my = (y >= y_lo) & (y <= y_hi)
sub_x = x[mx]
sub_y = y[my]
sub_alpha = slice_xy[np.ix_(mx, my)]

fig, ax = plt.subplots(figsize=(10, 7))
pc = ax.pcolormesh(
    sub_x / D, sub_y / D, sub_alpha.T,
    shading="nearest",
    cmap="viridis",
    vmin=0.0, vmax=1.0,
    edgecolors="k",
    linewidths=0.4,
)
ax.contour(sub_x / D, sub_y / D, sub_alpha.T, levels=[0.5], colors="r", linewidths=1.5)

for i, xv in enumerate(sub_x):
    for j, yv in enumerate(sub_y):
        a_val = float(sub_alpha[i, j])
        if 0.05 < a_val < 0.95:
            txt_color = "w" if a_val < 0.5 else "k"
            ax.text(xv / D, yv / D, f"{a_val:.2f}",
                    ha="center", va="center", fontsize=6, color=txt_color)

ax.set_xlabel("x / D")
ax.set_ylabel("y / D")
ax.set_aspect("equal")
ax.set_title(f"Right-droplet outer face, z=0 (step {step})")
fig.colorbar(pc, ax=ax, label=r"$\alpha_1$")
plt.show()

n_interface = int(((sub_alpha > 0.05) & (sub_alpha < 0.95)).sum())
print(f"Interface cells in this close-up (0.05 < alpha < 0.95): {n_interface}")

## $\alpha_1$ along an x-ray through each droplet centre

One data point per cell. Step line uses `drawstyle="steps-mid"` so the horizontal segment at each cell centre has length $\Delta x$. Faint vertical lines are cell boundaries. Dotted horizontals at 0.05 and 0.95 show what the thickness extraction will interpolate between.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (label, (xc, yc, zc)) in zip(axes, DROPLETS.items()):
    try:
        _, _, x_seg, a_seg, x_05, x_95 = interface_thickness_x_ray(alpha, x, y, z, xc, yc, zc, R)
        have_crossings = True
    except RuntimeError as err:
        have_crossings = False
        msg = str(err)
        jy = int(np.argmin(np.abs(y - yc)))
        kz = int(np.argmin(np.abs(z - zc)))
        if xc >= 0:
            mask = (x >= xc) & (x <= xc + 2.0 * R)
        else:
            mask = (x <= xc) & (x >= xc - 2.0 * R)
        x_seg = x[mask]
        a_seg = alpha[mask, jy, kz]

    xrel = (x_seg - xc) / D
    ax.plot(xrel, a_seg, drawstyle="steps-mid", lw=1.2, color="C0")
    ax.plot(xrel, a_seg, "o", color="C0", ms=4)

    edges = np.concatenate((
        [x_seg[0] - 0.5 * (x_seg[1] - x_seg[0])],
        0.5 * (x_seg[:-1] + x_seg[1:]),
        [x_seg[-1] + 0.5 * (x_seg[-1] - x_seg[-2])],
    ))
    for e in edges:
        ax.axvline((e - xc) / D, color="k", lw=0.2, alpha=0.3)

    ax.axhline(0.05, color="k", lw=0.5, ls=":")
    ax.axhline(0.95, color="k", lw=0.5, ls=":")

    if have_crossings:
        ax.plot([(x_05 - xc) / D, (x_95 - xc) / D], [0.05, 0.95], "x", color="r", ms=10, mew=2,
                label="interp crossings")
        ax.legend(loc="best", fontsize=8)

    ax.set_xlabel(r"$(x - x_c) / D$")
    ax.set_title(label if have_crossings else f"{label} — no 0.05/0.95 crossing on this ray")
    ax.grid(alpha=0.2)

axes[0].set_ylabel(r"$\alpha_1$")
fig.suptitle(f"$\\alpha_1$ along x-ray through droplet centre (step {step})")
plt.tight_layout()
plt.show()

## Derived: interface thickness

Two independent estimates of interface thickness, derived from the field you just looked at:
- **Ray method** — $|x(\alpha=0.95) - x(\alpha=0.05)|$ interpolated along the x-ray through each droplet centre.
- **$|\nabla \alpha|$ method** — $0.9 / \max |\nabla \alpha|$ over a 1.5R cube around each droplet. Assumes an approximately linear transition.

If these two disagree by more than a factor of ~2, the interface is not well-represented by an axis-aligned ray (e.g. tilted interface, or the ray misses the interface).

In [ ]:
results = {}
for label, (xc, yc, zc) in DROPLETS.items():
    try:
        t_m, t_c, *_ = interface_thickness_x_ray(alpha, x, y, z, xc, yc, zc, R)
        results[label] = (t_m, t_c)
    except RuntimeError as err:
        print(f"  {label:>5}: ray method failed — {err}")
        results[label] = (np.nan, np.nan)

print("Ray method (0.05 <= alpha_1 <= 0.95):")
for k, (t_m, t_c) in results.items():
    if not np.isnan(t_c):
        print(f"  {k:>5}: {t_c:.2f} cells = {t_m*1e6:.2f} um")
vals = [v[1] for v in results.values() if not np.isnan(v[1])]
t_mean_cells = float(np.mean(vals)) if vals else float("nan")
print(f"  mean : {t_mean_cells:.2f} cells")

print()
print("|grad alpha| method (max over a 1.5R cube):")
for label, (xc, yc, zc) in DROPLETS.items():
    mx = (x >= xc - 1.5 * R) & (x <= xc + 1.5 * R)
    my = (y >= yc - 1.5 * R) & (y <= yc + 1.5 * R)
    mz = (z >= zc - 1.5 * R) & (z <= zc + 1.5 * R)
    a_box = alpha[np.ix_(mx, my, mz)]
    gx_box = np.gradient(a_box, x[mx], axis=0)
    gy_box = np.gradient(a_box, y[my], axis=1)
    gz_box = np.gradient(a_box, z[mz], axis=2)
    gmag = np.sqrt(gx_box**2 + gy_box**2 + gz_box**2)
    t_grad_m = 0.9 / gmag.max()
    dx_local = float(np.mean(np.diff(x[mx])))
    print(f"  {label:>5}: max|grad alpha|={gmag.max():.3g} 1/m  ->  {t_grad_m*1e6:.2f} um ({t_grad_m/dx_local:.2f} cells)")

In [ ]:
if len(steps) < 2:
    print("Only one step available — skipping time-series.")
else:
    series = {k: [] for k in DROPLETS}
    for s in steps:
        d_s = assemble_silo(CASE_DIR, s)
        a_s = d_s.variables[ALPHA_NAME]
        xs, ys, zs = d_s.x_cc, d_s.y_cc, d_s.z_cc
        for label, (xc0, _yc0, _zc0) in DROPLETS.items():
            mx = (xs >= xc0 - 1.5 * R) & (xs <= xc0 + 1.5 * R)
            w = a_s[mx, :, :]
            mass = float(w.sum())
            if mass <= 0:
                series[label].append(np.nan)
                continue
            xw = float(np.sum(w.sum(axis=(1, 2)) * xs[mx]) / mass)
            yw = float(np.sum(w.sum(axis=(0, 2)) * ys) / mass)
            zw = float(np.sum(w.sum(axis=(0, 1)) * zs) / mass)
            try:
                _, t_c, *_ = interface_thickness_x_ray(a_s, xs, ys, zs, xw, yw, zw, R)
            except RuntimeError:
                t_c = np.nan
            series[label].append(t_c)

    fig, ax = plt.subplots(figsize=(7, 4))
    for label, vals in series.items():
        ax.plot(steps, vals, "o-", label=label)
    ax.set_xlabel("step")
    ax.set_ylabel("interface thickness [cells]")
    ax.set_title("Interface thickness over time")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()

In [ ]:
print(f"case: {CASE_LABEL}   step: {step}")
print(f"grid: {stats['x'][0]} x {stats['y'][0]} x {stats['z'][0]}")
print(f"CPD : x={D/dx:.1f}  y={D/dy:.1f}  z={D/dz:.1f}")
for k, (t_m, t_c) in results.items():
    if not np.isnan(t_c):
        print(f"interface thickness ({k:>5}): {t_c:.2f} cells = {t_m*1e6:.2f} um")
if not np.isnan(t_mean_cells):
    print(f"mean interface thickness: {t_mean_cells:.2f} cells")